# Chapitre 5 — Nettoyage des données

## 5.4 Traitement des valeurs aberrantes (Outliers)

---

### Objectifs

À la fin de cette leçon, vous serez capable de :
- **Identifier** les valeurs aberrantes avec différentes méthodes
- **Distinguer** les erreurs (à supprimer) des valeurs extrêmes réelles (à conserver)
- **Appliquer** des règles métier pour supprimer les valeurs impossibles

---

## 5.4.1 Qu'est-ce qu'un outlier ?

Un **outlier** (valeur aberrante) est une valeur qui s'écarte significativement des autres observations.

```
┌─────────────────────────────────────────────────────────────────────┐
│                    TYPES D'OUTLIERS                                 │
├─────────────────────┬───────────────────────────────────────────────┤
│      ERREUR         │  Valeur IMPOSSIBLE                            │
│                     │  Ex: âge = -5, âge = 350                      │
│                     │  → SUPPRIMER (règle métier fixe)              │
├─────────────────────┼───────────────────────────────────────────────┤
│   VALEUR EXTRÊME    │  Valeur POSSIBLE mais rare                    │
│      RÉELLE         │  Ex: salaire PDG = 10M€                       │
│                     │  → GARDER ou traiter en Module 3              │
├─────────────────────┼───────────────────────────────────────────────┤
│   VALEUR CIBLE      │  L'outlier EST ce qu'on cherche !             │
│   (Détection)       │  Ex: fraude, anomalie                         │
│                     │  → NE JAMAIS SUPPRIMER                        │
└─────────────────────┴───────────────────────────────────────────────┘
```

**Question :** Si vous analysez des transactions bancaires pour détecter des fraudes, devez-vous supprimer une transaction de 50 000€ alors que la moyenne est de 50€ ?

*(Réponse attendue : Non ! C'est peut-être exactement la fraude que vous cherchez)*

---

## 5.4.2 Identifier les outliers

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Données avec outliers
np.random.seed(42)
df = pd.DataFrame({
    'id': range(100),
    'age': np.concatenate([np.random.randint(20, 65, 97), [-5, 150, 200]]),
    'salaire': np.concatenate([np.random.normal(50000, 10000, 96), [500000, -1000, 48000, 52000]])
})

print("Statistiques descriptives :")
print(df.describe())

In [ ]:
# Méthode 1 : Visualisation avec boxplot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['age'].plot(kind='box', ax=axes[0], title='Distribution des âges')
df['salaire'].plot(kind='box', ax=axes[1], title='Distribution des salaires')

plt.tight_layout()
plt.show()

print("→ Les points au-delà des moustaches sont des outliers potentiels")

In [ ]:
# Méthode 2 : Statistiques min/max
print("Valeurs extrêmes :")
print(f"Age min: {df['age'].min()}, max: {df['age'].max()}")
print(f"Salaire min: {df['salaire'].min()}, max: {df['salaire'].max()}")

print("\n→ Age = -5 et 200 sont clairement impossibles")
print("→ Salaire = -1000 est impossible (négatif)")

In [ ]:
# Méthode 3 : IQR (pour DÉTECTION seulement)
Q1 = df['salaire'].quantile(0.25)
Q3 = df['salaire'].quantile(0.75)
IQR = Q3 - Q1

borne_inf = Q1 - 1.5 * IQR
borne_sup = Q3 + 1.5 * IQR

print(f"Méthode IQR :")
print(f"  Q1 = {Q1:.0f}, Q3 = {Q3:.0f}, IQR = {IQR:.0f}")
print(f"  Bornes théoriques : [{borne_inf:.0f}, {borne_sup:.0f}]")

outliers = df[(df['salaire'] < borne_inf) | (df['salaire'] > borne_sup)]
print(f"\nOutliers détectés ({len(outliers)}) :")
print(outliers)

---

## 5.4.3 Supprimer les valeurs IMPOSSIBLES (règles fixes)

En nettoyage de données, nous ne supprimons que les valeurs **clairement impossibles** selon des règles métier connues à l'avance.

| Colonne | Règle métier | Valeur impossible |
|---------|--------------|-------------------|
| `age` | 0 ≤ age ≤ 120 | age = -5, age = 200 |
| `salaire` | salaire ≥ 0 | salaire = -1000 |
| `taille_cm` | 50 ≤ taille ≤ 250 | taille = 500 |
| `temperature_C` | -50 ≤ T ≤ 60 | T = 150 |

In [ ]:
# Supprimer les valeurs impossibles avec des règles FIXES
print(f"Avant nettoyage : {len(df)} lignes")

# Règle 1 : age >= 0
df_clean = df[df['age'] >= 0]
print(f"Après age >= 0 : {len(df_clean)} lignes")

# Règle 2 : age <= 120
df_clean = df_clean[df_clean['age'] <= 120]
print(f"Après age <= 120 : {len(df_clean)} lignes")

# Règle 3 : salaire >= 0
df_clean = df_clean[df_clean['salaire'] >= 0]
print(f"Après salaire >= 0 : {len(df_clean)} lignes")

In [ ]:
# Version compacte avec query()
df_clean = df.query('age >= 0 and age <= 120 and salaire >= 0')
print(f"Avec query() : {len(df_clean)} lignes")

# Vérification
print(f"\nNouvelles statistiques :")
print(df_clean.describe())

In [ ]:
# Fonction réutilisable pour appliquer des règles métier
def appliquer_regles_metier(df, regles):
    """
    Applique des règles métier fixes pour supprimer les valeurs impossibles.
    
    Args:
        df: DataFrame
        regles: dict de la forme {'colonne': (min, max)}
    
    Returns:
        DataFrame nettoyé
    """
    df_clean = df.copy()
    for col, (val_min, val_max) in regles.items():
        avant = len(df_clean)
        df_clean = df_clean[(df_clean[col] >= val_min) & (df_clean[col] <= val_max)]
        apres = len(df_clean)
        if avant != apres:
            print(f"  {col}: {avant - apres} lignes supprimées (hors [{val_min}, {val_max}])")
    return df_clean

# Utilisation
regles = {
    'age': (0, 120),
    'salaire': (0, float('inf'))  # Pas de limite max pour le salaire
}

print("Application des règles métier :")
df_final = appliquer_regles_metier(df, regles)
print(f"\nRésultat : {len(df_final)} lignes")

---

## 5.4.4 Et les valeurs extrêmes mais RÉELLES ?

Après avoir supprimé les valeurs **impossibles**, il peut rester des valeurs **extrêmes mais possibles** :
- Un salaire de 500 000€ (rare mais réel)
- Un âge de 95 ans (rare mais possible)

**Que faire avec ces valeurs ?**

```
┌─────────────────────────────────────────────────────────────────────┐
│  En Module 2 (maintenant) : GARDER ces valeurs                      │
│                                                                     │
│  En Module 3 (ML Pipeline) : Options de traitement :                │
│  • OutlierCapper (winsorisation avec bornes du train)               │
│  • Log transformation                                               │
│  • Robuste scaler                                                   │
│  • Les garder (certains modèles les gèrent bien)                    │
└─────────────────────────────────────────────────────────────────────┘
```

> 💡 **Pourquoi ne pas winsoriser maintenant ?** La winsorisation calcule des percentiles sur les données. Si vous le faites avant le train/test split, vous incluez les données de test → **data leakage**.

---

## ✍️ Exercice 5.4 : Identifier et nettoyer les outliers

In [ ]:
import pandas as pd
import numpy as np

# Dataset avec des outliers
np.random.seed(42)
df_ex = pd.DataFrame({
    'patient_id': range(500),
    'age': np.concatenate([
        np.random.randint(18, 90, 495),
        [-10, 0, 150, 200, 999]  # Outliers
    ]),
    'poids_kg': np.concatenate([
        np.random.normal(70, 15, 495),
        [-50, 0, 300, 400, 500]  # Outliers
    ]),
    'taille_cm': np.concatenate([
        np.random.normal(170, 10, 495),
        [0, 10, 50, 300, 350]  # Outliers
    ])
})

print("Statistiques :")
print(df_ex.describe())

In [ ]:
# 1. Identifiez les valeurs impossibles
print("Valeurs min/max :")
print(f"  age: [{df_ex['age'].min()}, {df_ex['age'].max()}]")
print(f"  poids: [{df_ex['poids_kg'].min():.1f}, {df_ex['poids_kg'].max():.1f}]")
print(f"  taille: [{df_ex['taille_cm'].min():.1f}, {df_ex['taille_cm'].max():.1f}]")

In [ ]:
# 2. Définissez les règles métier
regles_patients = {
    'age': (0, 120),           # Age humain possible
    'poids_kg': (2, 300),      # Poids humain possible (nouveau-né à obèse)
    'taille_cm': (50, 250)     # Taille humaine possible
}

# 3. Appliquez les règles
print(f"Avant nettoyage : {len(df_ex)} lignes")
df_clean = appliquer_regles_metier(df_ex, regles_patients)
print(f"\nAprès nettoyage : {len(df_clean)} lignes")

In [ ]:
# 4. Vérification
print("Nouvelles statistiques :")
print(df_clean.describe())

---

## 📝 Résumé

| Type d'outlier | Action en Module 2 | Action en Module 3 |
|----------------|--------------------|--------------------||
| **Valeur impossible** (age = -5) | ✅ SUPPRIMER (règle fixe) | - |
| **Valeur extrême réelle** (salaire = 500k) | ❌ GARDER | Winsorisation dans Pipeline |
| **Valeur cible** (fraude) | ❌ NE JAMAIS SUPPRIMER | - |

**Règles de nettoyage sûres (Module 2) :**
- `df[df['age'] >= 0]` → règle métier fixe
- `df[df['age'] <= 120]` → règle métier fixe
- `df.query('salaire >= 0')` → règle métier fixe

---

## ➡️ Et le capping (winsorisation) ?

La winsorisation (ramener les valeurs aux percentiles 5% et 95%) sera enseignée dans le **Module 3** car :
- Elle calcule des statistiques (percentiles) sur les données
- Ces statistiques doivent être calculées sur le **train set uniquement**
- Sinon → **data leakage**

```
┌─────────────────────────────────────────────────────────────────────┐
│  Module 2 (maintenant)         │    Module 3 (Pipeline ML)         │
├────────────────────────────────┼───────────────────────────────────┤
│  • Supprimer valeurs           │    • OutlierCapper dans Pipeline  │
│    impossibles (règles fixes)  │    • Fit sur train, transform all │
│  • Garder valeurs extrêmes     │    • Pas de data leakage          │
│    réelles                     │                                   │
└────────────────────────────────┴───────────────────────────────────┘
```